# CorpusHop — Compositional Pipeline

This notebook is a reusable version of the **3-hop Compositional** pipeline.

### What you need to change
Usually, only edit the **Configuration** cell below:

- `INPUT_DATA`: a JSON/JSONL file **or a directory** containing JSON/JSONL files.
- `FIELD_MAP`: only needed when your field names are unusual.
- `OUTPUT_DIR`: where cleaned data, graph files, caches, and generated questions are written.
- model / generation settings.

### Minimum input requirements
Each chunk needs:

1. a text field, e.g. `text`, `content`, or `context`
2. a triple field containing `[subject, relation, object]` triples, e.g. `extracts` or `triples`

Chunk IDs are optional; if absent, IDs are generated. `entities` and `topic` are optional.

The loader automatically recognizes several common field names and converts all data to the canonical schema used internally:

```json
{
  "id": "chunk_001",
  "topic": "optional-topic",
  "text": "chunk text",
  "extracts": [["subject", "relation", "object"]],
  "entities": ["subject", "object"]
}
```

### Compositional structure
The reasoning logic remains the same as the original notebook:

- `t1 = (s1, r1, s2)` from chunk **A**
- `t2 = (s4, r2, s2)` from chunk **B**
- `t3 = (s2, r3, s3)` from chunk **C**
- graph structure: **A → C ← B**
- greedy selection minimizes `deg(A) + deg(B) + deg(C)`
- **PASS** removes A, B, and C (node-disjoint acceptance)
- **FAIL** removes only edges A→C and B→C

The graph still uses exact normalized object→subject matching; no embedding or fuzzy matching is introduced.


## Dependencies and API key

Install the required packages once:

```bash
pip install openai python-dotenv networkx numpy scipy
```

Set your OpenAI key without committing it to GitHub:

```bash
# .env
OPENAI_API_KEY=your_key_here
```

Recommended `.gitignore` entries:

```text
.env
outputs/
__pycache__/
.ipynb_checkpoints/
```

For a custom dataset, you should normally only need to change `INPUT_DATA` and, if auto-detection cannot recognize your field names, `FIELD_MAP`.

> For independent datasets or experiments, use a separate `OUTPUT_DIR` so resume files from different runs are not mixed.


In [ ]:
from pathlib import Path

# =============================================================================
# CONFIGURATION — normally this is the only cell you need to edit
# =============================================================================

# Input may be:
#   1) one .json / .jsonl / .ndjson file, OR
#   2) a directory containing those files.
INPUT_DATA = Path("./data/chunks.jsonl")

# Use a different OUTPUT_DIR for each dataset/experiment when you want to keep
# independent resume state and generated-question files.
OUTPUT_DIR = Path("./outputs")

# Optional field mapping.
# Use None to auto-detect common aliases.
# Example for a custom dataset:
# FIELD_MAP = {
#     "id": "chunk_key",
#     "text": "passage",
#     "triples": "spo_list",
#     "entities": None,
#     "topic": "category",
# }
FIELD_MAP = {
    "id": None,
    "text": None,
    "triples": None,
    "entities": None,
    "topic": None,
}

# Search nested directories when INPUT_DATA is a directory.
RECURSIVE_INPUT = True

# Generated filenames. Usually you do not need to change these.
CLEANED_FILENAME = "docs_with_triples_cleaned.json"
GRAPH_FILENAME = "graph_data_exact.pkl"
CONNECTIONS_FILENAME = "node_connections_exact.txt"

# OpenAI settings.
# Put OPENAI_API_KEY in your environment or in a local .env file.
GENERATION_MODEL = "gpt-4o-mini"
JUDGE_MODEL_NAME = "gpt-4o-mini"

# Preserve the original Compositional generation setting by default.
GENERATION_TEMPERATURE = 0.5

# None = continue until no valid path remains.
MAX_SELECTED_PATHS = None

# SQLite batching settings. Increase these only if you know you need to tune I/O.
DOC_DB_BATCH_SIZE = 5_000
TRIPLE_DB_BATCH_SIZE = 10_000

# Execution switches.
# Cleaning + graph building do not call an LLM.
RUN_CLEANING = True
RUN_GRAPH_BUILDING = True

# This stage calls the OpenAI API and can incur cost.
# Set to True when you are ready to generate and judge questions.
RUN_LLM_PIPELINE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input : {INPUT_DATA}")
print(f"Output: {OUTPUT_DIR.resolve()}")


## 1. Generic input loader and canonical schema


In [ ]:
import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Iterable


SUPPORTED_EXTENSIONS = {".json", ".jsonl", ".ndjson", ".txt"}

FIELD_ALIASES = {
    "id": (
        "id", "chunk_id", "chunkid", "doc_id", "document_id",
        "passage_id", "section_id", "uid",
    ),
    "text": (
        "text", "content", "context", "passage", "chunk_text",
        "document", "body",
    ),
    "triples": (
        "extracts", "triples", "relations", "spo", "spo_triples",
        "knowledge_triples",
    ),
    "entities": (
        "entities", "entity_list", "named_entities",
    ),
    "topic": (
        "topic", "topic_id", "category", "domain", "group",
    ),
}


def _pick_field(record: dict, canonical_name: str, field_map: dict) -> str | None:
    """Resolve one canonical field to an actual key in a record."""
    explicit = field_map.get(canonical_name)
    if explicit:
        if explicit not in record:
            raise KeyError(
                f"FIELD_MAP maps '{canonical_name}' to '{explicit}', "
                f"but that key is missing. Available keys: {list(record.keys())}"
            )
        return explicit

    for candidate in FIELD_ALIASES[canonical_name]:
        if candidate in record:
            return candidate
    return None


def _iter_input_files(source: str | Path, recursive: bool = True) -> list[Path]:
    source = Path(source)

    if source.is_file():
        if source.suffix.lower() not in SUPPORTED_EXTENSIONS:
            raise ValueError(
                f"Unsupported input type: {source.suffix}. "
                f"Supported: {sorted(SUPPORTED_EXTENSIONS)}"
            )
        return [source]

    if not source.is_dir():
        raise FileNotFoundError(f"Input file/directory not found: {source}")

    iterator = source.rglob("*") if recursive else source.glob("*")
    files = sorted(
        p for p in iterator
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    if not files:
        raise FileNotFoundError(
            f"No supported data files found under {source}. "
            f"Supported: {sorted(SUPPORTED_EXTENSIONS)}"
        )
    return files


def _load_records_from_file(path: Path) -> list[dict]:
    """
    Load either:
      - JSON array: [{...}, {...}]
      - JSON object containing a common list key such as data/documents/chunks/records
      - JSONL / NDJSON / line-delimited JSON
    """
    suffix = path.suffix.lower()

    if suffix == ".json":
        with path.open("r", encoding="utf-8") as fh:
            obj = json.load(fh)

        if isinstance(obj, list):
            records = obj
        elif isinstance(obj, dict):
            records = None
            for key in ("data", "documents", "docs", "chunks", "records", "items"):
                if isinstance(obj.get(key), list):
                    records = obj[key]
                    break
            if records is None:
                # Treat one JSON object as one record.
                records = [obj]
        else:
            raise ValueError(f"Unsupported JSON structure in {path}")

        if not all(isinstance(x, dict) for x in records):
            raise ValueError(f"All records must be JSON objects in {path}")
        return records

    # .jsonl / .ndjson / .txt -> one JSON object per non-empty line.
    records = []
    with path.open("r", encoding="utf-8") as fh:
        for line_number, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON at {path}:{line_number}: {exc}"
                ) from exc
            if not isinstance(record, dict):
                raise ValueError(
                    f"Expected a JSON object at {path}:{line_number}, "
                    f"got {type(record).__name__}"
                )
            records.append(record)
    return records


def _canonicalize_record(
    record: dict,
    generated_id: str,
    field_map: dict,
    source_file: str,
    source_index: int,
) -> dict:
    """Convert a source record to CorpusHop's internal canonical schema."""
    id_key = _pick_field(record, "id", field_map)
    text_key = _pick_field(record, "text", field_map)
    triple_key = _pick_field(record, "triples", field_map)
    entities_key = _pick_field(record, "entities", field_map)
    topic_key = _pick_field(record, "topic", field_map)

    if text_key is None:
        raise KeyError(
            f"Could not find a text field in {source_file} record #{source_index}. "
            f"Set FIELD_MAP['text']. Available keys: {list(record.keys())}"
        )
    if triple_key is None:
        raise KeyError(
            f"Could not find a triple field in {source_file} record #{source_index}. "
            f"Set FIELD_MAP['triples']. Available keys: {list(record.keys())}"
        )

    raw_id = record.get(id_key) if id_key else generated_id
    chunk_id = str(raw_id).strip() if raw_id is not None else generated_id
    if not chunk_id:
        chunk_id = generated_id

    text = record.get(text_key, "")
    if not isinstance(text, str):
        text = str(text)

    triples = record.get(triple_key, [])
    if triples is None:
        triples = []
    if not isinstance(triples, list):
        raise TypeError(
            f"Triple field must be a list in {source_file} record #{source_index}"
        )

    entities = record.get(entities_key, []) if entities_key else []
    if entities is None:
        entities = []
    if not isinstance(entities, list):
        entities = [str(entities)]

    topic = record.get(topic_key, "") if topic_key else ""
    if topic is None:
        topic = ""

    return {
        "id": chunk_id,
        "topic": str(topic),
        "text": text,
        "extracts": triples,
        "entities": [str(e) for e in entities if str(e).strip()],
        "_source_file": source_file,
        "_source_index": source_index,
    }


def load_canonical_documents(
    source: str | Path,
    field_map: dict | None = None,
    recursive: bool = True,
) -> list[dict]:
    """
    Load one file or a directory of files and normalize records to:
      id, topic, text, extracts, entities

    IDs are generated when absent.
    """
    field_map = field_map or {}
    files = _iter_input_files(source, recursive=recursive)

    documents = []
    seen_ids = set()
    global_index = 0

    print(f"Found {len(files)} input file(s).")

    for file_path in files:
        records = _load_records_from_file(file_path)
        print(f"  {file_path}: {len(records):,} record(s)")

        for local_index, record in enumerate(records, 1):
            global_index += 1
            generated_id = f"chunk_{global_index:08d}"
            doc = _canonicalize_record(
                record=record,
                generated_id=generated_id,
                field_map=field_map,
                source_file=str(file_path),
                source_index=local_index,
            )

            if doc["id"] in seen_ids:
                raise ValueError(
                    f"Duplicate chunk id '{doc['id']}'. "
                    "IDs must be unique across all input files."
                )
            seen_ids.add(doc["id"])
            documents.append(doc)

    print(f"Loaded {len(documents):,} total chunk(s).")
    return documents


def _norm(s: str) -> str:
    """
    Unicode-aware exact-match normalization.

    Matching remains exact after normalization:
      - Unicode NFKC normalization
      - case-folding
      - punctuation/symbols -> spaces
      - whitespace collapse

    Letters and digits from any language are retained.
    """
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    chars = []
    for ch in s:
        category = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        elif ch.isspace() or category[0] in {"P", "S"}:
            chars.append(" ")
        else:
            chars.append(" ")
    return " ".join("".join(chars).split())


## 2. Clean triples and keep eligible chunks


In [ ]:
def is_valid_triplet(triplet) -> bool:
    """Return True only for a well-formed [subject, relation, object] triple."""
    if not isinstance(triplet, (list, tuple)) or len(triplet) != 3:
        return False

    subject, relation, obj = triplet
    if not all(isinstance(x, str) and x.strip() for x in (subject, relation, obj)):
        return False

    # Drop self-loops.
    if subject.strip() == obj.strip():
        return False

    return True


def clean_dataset(
    input_source: str | Path,
    output_path: str | Path,
    field_map: dict | None = None,
    recursive: bool = True,
) -> list[dict]:
    """
    Load arbitrary supported input, normalize its schema, clean triples,
    derive entities from surviving triples, and write canonical JSON.
    """
    output_path = Path(output_path)
    raw_docs = load_canonical_documents(
        input_source,
        field_map=field_map,
        recursive=recursive,
    )

    stats = {
        "original_docs": len(raw_docs),
        "docs_after_cleaning": 0,
        "original_total_triplets": 0,
        "cleaned_total_triplets": 0,
        "removed_triplets": 0,
    }

    cleaned_data = []

    for doc in raw_docs:
        original_triplets = doc.get("extracts", [])
        stats["original_total_triplets"] += len(original_triplets)

        valid_triplets = []
        for triplet in original_triplets:
            if is_valid_triplet(triplet):
                valid_triplets.append([str(x).strip() for x in triplet])
            else:
                stats["removed_triplets"] += 1

        # Eligible chunks must contain at least one valid triple.
        if not valid_triplets:
            continue

        # Rebuild entities from the cleaned triples so an input `entities`
        # field is optional.
        valid_entities = sorted(
            {t[0] for t in valid_triplets} | {t[2] for t in valid_triplets}
        )

        cleaned_doc = dict(doc)
        cleaned_doc["extracts"] = valid_triplets
        cleaned_doc["entities"] = valid_entities
        cleaned_data.append(cleaned_doc)

        stats["cleaned_total_triplets"] += len(valid_triplets)

    stats["docs_after_cleaning"] = len(cleaned_data)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as fh:
        json.dump(cleaned_data, fh, ensure_ascii=False, indent=2)

    print("\nData Cleaning Statistics")
    print("=" * 52)
    print(f"Input                  : {input_source}")
    print(f"Output                 : {output_path}")
    print("-" * 52)
    print(f"Original documents     : {stats['original_docs']:,}")
    print(f"Documents after clean  : {stats['docs_after_cleaning']:,}")
    print(f"Original total triples : {stats['original_total_triplets']:,}")
    print(f"Removed invalid triples: {stats['removed_triplets']:,}")
    print(f"Cleaned total triples  : {stats['cleaned_total_triplets']:,}")
    print("=" * 52)

    return cleaned_data


## 3. Build exact-match chunk graph


In [ ]:
import pickle
import time
from collections import defaultdict

import networkx as nx


def create_graph_and_nodes(
    cleaned_file: str | Path,
    output_folder: str | Path,
    graph_filename: str = GRAPH_FILENAME,
    connections_filename: str = CONNECTIONS_FILENAME,
):
    """
    Build the directed chunk graph.

    Edge A -> B exists when a normalized object from a triple in A exactly
    equals a normalized subject from a triple in B.

    No embeddings, fuzzy matching, or semantic similarity are used.
    """
    cleaned_file = Path(cleaned_file)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    if not cleaned_file.exists():
        raise FileNotFoundError(
            f"Cleaned dataset not found: {cleaned_file}. "
            "Run clean_dataset(...) first."
        )

    with cleaned_file.open("r", encoding="utf-8") as fh:
        data = json.load(fh)

    if not isinstance(data, list):
        raise ValueError("Cleaned dataset must be a JSON array.")

    print(f"Loaded {len(data):,} cleaned documents")
    print("Matching strategy: exact normalized object -> subject\n")

    # norm_subject -> list[(chunk_id, triple)]
    print("Building subject index...")
    subj_index = defaultdict(list)

    for doc in data:
        cid = str(doc["id"])
        for triple in doc.get("extracts", []):
            if len(triple) < 3:
                continue
            ns = _norm(triple[0])
            if ns:
                subj_index[ns].append((cid, triple))

    print(f"  Unique normalized subjects: {len(subj_index):,}")

    G = nx.DiGraph()
    for doc in data:
        cid = str(doc["id"])
        G.add_node(cid, **doc)
    print(f"  Nodes: {G.number_of_nodes():,}")

    print("\nCreating edges...")
    edge_details = {}
    t0 = time.time()

    for doc_idx, src_doc in enumerate(data):
        if doc_idx and doc_idx % 300 == 0:
            elapsed = max(time.time() - t0, 1e-6)
            rate = doc_idx / elapsed
            remaining = len(data) - doc_idx
            eta = remaining / rate if rate else 0
            print(
                f"  {doc_idx:,}/{len(data):,}  "
                f"edges={G.number_of_edges():,}  ETA={eta/60:.1f} min"
            )

        src_cid = str(src_doc["id"])

        for src_triple in src_doc.get("extracts", []):
            if len(src_triple) < 3:
                continue

            obj_norm = _norm(src_triple[2])
            if not obj_norm:
                continue

            for dst_cid, dst_triple in subj_index.get(obj_norm, []):
                if dst_cid == src_cid:
                    continue

                key = (src_cid, dst_cid)
                pair = {
                    "src_triple": src_triple,
                    "dst_triple": dst_triple,
                    "entity": obj_norm,
                }

                if key not in edge_details:
                    edge_details[key] = {
                        "src_id": src_cid,
                        "dst_id": dst_cid,
                        "link_entity": obj_norm,
                        "score": 1.0,
                        "pairs": [pair],
                    }
                    G.add_edge(
                        src_cid,
                        dst_cid,
                        link_entity=obj_norm,
                        score=1.0,
                        num_pairs=1,
                    )
                else:
                    edge_details[key]["pairs"].append(pair)
                    G[src_cid][dst_cid]["num_pairs"] += 1

    elapsed = time.time() - t0
    print(f"\nEdge creation done in {elapsed:.1f}s")
    print(f"  Nodes: {G.number_of_nodes():,}")
    print(f"  Edges: {G.number_of_edges():,}")

    print("\nSample convergent A -> C <- B structures:")
    shown = 0
    for c in G.nodes():
        preds = list(G.predecessors(c))
        if len(preds) < 2:
            continue
        for i in range(len(preds)):
            for j in range(i + 1, len(preds)):
                a, b = preds[i], preds[j]
                if len({a, b, c}) < 3:
                    continue
                ea, eb = G[a][c], G[b][c]
                print(
                    f"  {a} -['{ea['link_entity']}']-> {c} <-"
                    f"['{eb['link_entity']}']- {b}"
                )
                shown += 1
                if shown >= 5:
                    break
            if shown >= 5:
                break
        if shown >= 5:
            break
    if shown == 0:
        print("  (none)")

    connections_file = output_folder / connections_filename
    with connections_file.open("w", encoding="utf-8") as fh:
        fh.write("Directed Graph — exact normalized match\n")
        fh.write("=" * 60 + "\n")
        fh.write(f"Nodes : {G.number_of_nodes()}\n")
        fh.write(f"Edges : {G.number_of_edges()}\n\n")

        for (src, dst), d in edge_details.items():
            fh.write(
                f"{src} -> {dst} "
                f"[entity='{d['link_entity']}' pairs={len(d['pairs'])}]\n"
            )
            for pair in d["pairs"][:3]:
                s, t = pair["src_triple"], pair["dst_triple"]
                fh.write(f"  src: [{s[0]} | {s[1]} | {s[2]}]\n")
                fh.write(f"  dst: [{t[0]} | {t[1]} | {t[2]}]\n")
                fh.write(f"  matched entity: '{pair['entity']}'\n")
            if len(d["pairs"]) > 3:
                fh.write(f"  ... +{len(d['pairs']) - 3} more pairs\n")
            fh.write("\n")

    graph_file = output_folder / graph_filename
    with graph_file.open("wb") as fh:
        pickle.dump(
            {"G": G, "data": data, "edge_details": edge_details},
            fh,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    print(f"\nEdge list: {connections_file}")
    print(f"Graph    : {graph_file}")
    return G, data, edge_details, str(graph_file)


## 4. Greedy 3-hop Compositional selection, question generation, and LLM judge


In [ ]:
"""
multihop_pipeline.py  (convergent rewrite)
══════════════════════════════════════════
Unified 3-hop question generation pipeline — CONVERGENT (same-bridge) chain.

Chain shape  (A→C, B→C — two independent sources pointing to the same bridge):
  t1 = [s1, r1, s2]  from node_A   (s1 → s2)
  t2 = [s4, r2, s2]  from node_B   (s4 → s2, SAME s2 as t1, independent fact)
  t3 = [s2, r3, s3]  from node_C   (s2 → s3, final answer)

Score:  degree_sum = deg(A) + deg(B) + deg(C)   (undirected, within subgraph)

Lower-bound (MinK_conv):
  promised(A,B) = deg(A) + deg(B) + min_{C ∈ shared_succ(A,B)} deg(C)

Pairs are sorted by promised score ascending.  Once promised ≥ best_score,
the entire remaining component is pruned.

Skip set (SQLite-backed, in-memory for fast lookup):
  (A,B,C) combos with no valid triple chain are persisted across restarts.

chunk_to_doc (SQLite, not RAM):
  All doc data is stored in a `docs` table in triple_index.db and fetched
  on demand.  Startup cache stores only G.

Loop (one path at a time, fully resumable):
  1. Greedy-select the BEST remaining (A, B) → C convergent path.
  2. Generate Q1, Q2, Q3 and the combined Final question via the configured OpenAI model.
  3. Judge the Final question with an LLM judge.
  4. PASS  → save; remove all 3 nodes + incident edges from graph.
  5. FAIL  → remove only edges A→C and B→C; try the next best path.
  6. Repeat until no valid path remains or MAX_PATHS is reached.

Outputs  (all in <output_folder>/questions/)
──────────────────────────────────────────────
  multihop_questions_filtered.jsonl
  multihop_questions_filtered_readable.txt
  multihop_questions_all_results.jsonl
  multihop_questions_all_results_readable.txt
"""

import hashlib
import json
import os
import pickle
import re
import sqlite3
import time
import traceback
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import networkx as nx
import numpy as np
import scipy.sparse as sp
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ─────────────────────────────────────────────────────────────────────────────
# Config aliases from the notebook Configuration cell
# ─────────────────────────────────────────────────────────────────────────────

GEN_MODEL   = GENERATION_MODEL
JUDGE_MODEL = JUDGE_MODEL_NAME
TEMPERATURE = GENERATION_TEMPERATURE
MAX_PATHS   = MAX_SELECTED_PATHS

OUTPUT_FOLDER = str(OUTPUT_DIR)



# ─────────────────────────────────────────────────────────────────────────────
# Shared utilities
# ─────────────────────────────────────────────────────────────────────────────

def _norm(s: str) -> str:
    """
    Unicode-aware exact-match normalization shared with graph construction.

    Matching remains exact after normalization:
      - Unicode NFKC normalization
      - case-folding
      - punctuation/symbols -> spaces
      - whitespace collapse

    Letters and digits from any language are retained.
    """
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    chars = []
    for ch in s:
        category = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        elif ch.isspace() or category[0] in {"P", "S"}:
            chars.append(" ")
        else:
            chars.append(" ")
    return " ".join("".join(chars).split())


def _norm_exact(s: str) -> str:
    """Backward-compatible alias for the pipeline's exact normalizer."""
    return _norm(s)


def _words(e: str) -> list[str]:
    return re.findall(r"\w+", e.lower())


def _clean_ws(s: str) -> str:
    """
    Collapse all Unicode whitespace variants (non-breaking space \xa0,
    thin space, etc.) to a normal space, and trim.
    """
    return re.sub(r"\s+", " ", s).strip()


def _entities_differ(e1: str, e2: str, threshold: float = 0.95) -> bool:
    """True when e1 and e2 are meaningfully different entities."""
    if e1 == e2:
        return False
    n1 = re.sub(r"[^\w]", "", e1.lower())
    n2 = re.sub(r"[^\w]", "", e2.lower())
    if n1 == n2 or n1 in n2 or n2 in n1:
        return False
    w1, w2 = _words(e1), _words(e2)
    common  = set(w1) & set(w2)
    mn      = min(len(w1), len(w2))
    if mn > 0 and len(common) / mn >= threshold:
        return False
    return True


def _fmt(triple: list) -> str:
    if not triple or len(triple) < 3:
        return ""
    return f'["{triple[0]}", "{triple[1]}", "{triple[2]}"]'


def _parse_json(raw: str) -> dict:
    clean = (
        raw.strip()
           .removeprefix("```json")
           .removeprefix("```")
           .removesuffix("```")
           .strip()
    )
    return json.loads(clean)


# ─────────────────────────────────────────────────────────────────────────────
# Startup cache  (G only — chunk_to_doc lives in SQLite)
# ─────────────────────────────────────────────────────────────────────────────

_CACHE_KEYS = ("G",)


def _cache_dir(output_folder: str) -> str:
    results_folder = os.path.join(output_folder, "results")
    cache_dir      = os.path.join(results_folder, "startup_cache")
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir


def _legacy_cache_path(output_folder: str) -> str:
    return os.path.join(output_folder, "results", "startup_cache.pkl")


def _source_fingerprint(output_folder: str) -> str:
    """
    MD5 of size + content-hash of both source files — stale only if content
    actually changes.
    """
    graph_file   = os.path.join(output_folder, GRAPH_FILENAME)
    cleaned_file = os.path.join(output_folder, CLEANED_FILENAME)
    h = hashlib.md5()
    for f in [graph_file, cleaned_file]:
        if os.path.exists(f):
            st = os.stat(f)
            h.update(f"{f}:{st.st_size}".encode())
            with open(f, "rb") as fh:
                head = fh.read(1_048_576)
                h.update(head)
                if st.st_size > 1_048_576:
                    fh.seek(-1_048_576, os.SEEK_END)
                    h.update(fh.read(1_048_576))
    return h.hexdigest()


def _load_startup_cache(output_folder: str):
    """Returns G, or None if cache is missing/stale."""
    cache_dir   = _cache_dir(output_folder)
    fp_file     = os.path.join(cache_dir, "fingerprint.txt")
    shard_files = {k: os.path.join(cache_dir, f"{k}.pkl") for k in _CACHE_KEYS}
    expected_fp = _source_fingerprint(output_folder)

    if os.path.exists(fp_file) and all(os.path.exists(p) for p in shard_files.values()):
        try:
            with open(fp_file) as fh:
                if fh.read().strip() != expected_fp:
                    print("  Cache stale — rebuilding.")
                    return None
            t0 = time.perf_counter()
            print(f"Loading startup cache (split) from {cache_dir} ...")

            def _load_shard(key):
                with open(shard_files[key], "rb") as fh:
                    return key, pickle.load(fh)

            results: dict = {}
            with ThreadPoolExecutor(max_workers=len(_CACHE_KEYS)) as ex:
                for key, value in ex.map(_load_shard, _CACHE_KEYS):
                    results[key] = value

            G = results["G"]
            print(f"  Cache hit in {time.perf_counter()-t0:.1f}s — "
                  f"{G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges")
            return G
        except Exception as e:
            print(f"  Split cache load failed ({e}) — rebuilding.")
            return None

    # Legacy cache may still have chunk_to_doc / edge_details inside — load G only
    legacy = _legacy_cache_path(output_folder)
    if not os.path.exists(legacy):
        return None
    try:
        t0 = time.perf_counter()
        print(f"Loading startup cache (legacy) from {legacy} ...")
        with open(legacy, "rb") as fh:
            cached = pickle.load(fh)
        if cached.get("fingerprint") != expected_fp:
            print("  Cache stale — rebuilding.")
            return None
        G = cached["G"]
        print(f"  Cache hit in {time.perf_counter()-t0:.1f}s — "
              f"{G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges")
        try:
            _save_startup_cache(output_folder, G)
            os.remove(legacy)
            print("  Migrated legacy cache to split layout (G only).")
        except Exception as e:
            print(f"  (Could not migrate: {e})")
        return G
    except Exception as e:
        print(f"  Cache load failed ({e}) — rebuilding.")
        return None


def _save_startup_cache(output_folder: str, G) -> None:
    cache_dir = _cache_dir(output_folder)
    print(f"Saving startup cache → {cache_dir} ...")
    values = dict(G=G)
    total_bytes = 0
    for key in _CACHE_KEYS:
        shard = os.path.join(cache_dir, f"{key}.pkl")
        tmp   = shard + ".tmp"
        with open(tmp, "wb") as fh:
            pickle.dump(values[key], fh, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, shard)
        total_bytes += os.path.getsize(shard)
    fp_file = os.path.join(cache_dir, "fingerprint.txt")
    tmp_fp  = fp_file + ".tmp"
    with open(tmp_fp, "w") as fh:
        fh.write(_source_fingerprint(output_folder))
    os.replace(tmp_fp, fp_file)
    print(f"  Saved ({total_bytes/1_048_576:.0f} MB across {len(_CACHE_KEYS)} shards).")


# ─────────────────────────────────────────────────────────────────────────────
# Graph / data loading
# ─────────────────────────────────────────────────────────────────────────────

def _load_graph_and_data(output_folder: str) -> tuple:
    """Returns (G, raw_data_list_or_dict) — chunk_to_doc is NOT kept in RAM."""
    graph_file = os.path.join(output_folder, GRAPH_FILENAME)
    if not os.path.exists(graph_file):
        raise FileNotFoundError(graph_file)
    print(f"Loading graph from {graph_file} ...")
    t0 = time.perf_counter()
    with open(graph_file, "rb") as fh:
        gd = pickle.load(fh)
    G    = gd["G"]
    data = gd["data"]
    print(f"  {G.number_of_nodes():,} nodes  {G.number_of_edges():,} edges  "
          f"({time.perf_counter()-t0:.1f}s)")
    return G, data


# ─────────────────────────────────────────────────────────────────────────────
# Doc DB  +  Triple DB  +  Skip table  (all in triple_index.db)
# ─────────────────────────────────────────────────────────────────────────────

def _open_db(db_path: str) -> sqlite3.Connection:
    con = sqlite3.connect(db_path, check_same_thread=False)
    con.execute("PRAGMA journal_mode=WAL")
    con.execute("PRAGMA synchronous=NORMAL")
    con.execute("PRAGMA cache_size=-65536")
    con.execute("PRAGMA temp_store=MEMORY")
    return con


def _build_doc_db(data, con: sqlite3.Connection) -> None:
    """
    Store all doc fields in a `docs` table so chunk_to_doc never lives in RAM.
    Safe to call on an existing DB — checks `doc_db_ready` flag first.
    """
    con.execute("""
        CREATE TABLE IF NOT EXISTS docs (
            chunk_id  TEXT PRIMARY KEY,
            topic     TEXT,
            text      TEXT,
            entities  TEXT,
            extracts  TEXT
        )
    """)
    con.execute("CREATE TABLE IF NOT EXISTS doc_db_ready (done INTEGER)")

    already = con.execute("SELECT done FROM doc_db_ready").fetchone()
    if already:
        n = con.execute("SELECT COUNT(*) FROM docs").fetchone()[0]
        print(f"  Doc DB already built ({n:,} rows).")
        return

    if data is None:
        raise RuntimeError("Doc DB not built yet but data=None — pass raw data on first run.")

    print("Building doc DB ...")
    t0  = time.perf_counter()
    buf = []

    items = data.items() if isinstance(data, dict) else (
        (doc.get("chunk_id") or doc.get("id"), doc) for doc in data
    )

    for cid, doc in items:
        buf.append((
            cid,
            doc.get("topic",    ""),
            doc.get("text",     ""),
            json.dumps(doc.get("entities", []), ensure_ascii=False),
            json.dumps(doc.get("extracts",  []), ensure_ascii=False),
        ))
        if len(buf) >= DOC_DB_BATCH_SIZE:
            con.executemany("INSERT OR IGNORE INTO docs VALUES (?,?,?,?,?)", buf)
            buf.clear()
    if buf:
        con.executemany("INSERT OR IGNORE INTO docs VALUES (?,?,?,?,?)", buf)
    con.execute("INSERT INTO doc_db_ready VALUES (1)")
    con.commit()
    n = con.execute("SELECT COUNT(*) FROM docs").fetchone()[0]
    print(f"  Doc DB built: {n:,} rows  ({time.perf_counter()-t0:.1f}s)")


def _fetch_doc(con: sqlite3.Connection, chunk_id: str) -> dict:
    """Fetch a single doc by chunk_id from SQLite."""
    row = con.execute(
        "SELECT chunk_id, topic, text, entities, extracts FROM docs WHERE chunk_id=?",
        (chunk_id,)
    ).fetchone()
    if row is None:
        return {}
    cid, topic, text, entities_json, extracts_json = row
    return {
        "chunk_id": cid,
        "topic"   : topic or "",
        "text"    : text  or "",
        "entities": json.loads(entities_json) if entities_json else [],
        "extracts": json.loads(extracts_json) if extracts_json else [],
    }


def _build_triple_db(data, con: sqlite3.Connection) -> None:
    """
    Build the triples index table in SQLite.
    Iterates data once without keeping chunk_to_doc in RAM.
    """
    con.execute("""
        CREATE TABLE IF NOT EXISTS triples (
            chunk_id  TEXT NOT NULL,
            role      TEXT NOT NULL,
            key_norm  TEXT NOT NULL,
            s TEXT, r TEXT, o TEXT
        )
    """)
    con.execute("CREATE TABLE IF NOT EXISTS triple_db_ready (done INTEGER)")
    con.execute("CREATE INDEX IF NOT EXISTS idx_b ON triples (chunk_id, role, key_norm)")

    already = con.execute("SELECT done FROM triple_db_ready").fetchone()
    if already:
        n = con.execute("SELECT COUNT(*) FROM triples").fetchone()[0]
        print(f"  Triple DB already built ({n:,} rows).")
        return

    if data is None:
        raise RuntimeError("Triple DB not built yet but data=None — pass raw data on first run.")

    print("Building triple DB ...")
    t0  = time.perf_counter()
    buf = []

    items = data.items() if isinstance(data, dict) else (
        (doc.get("chunk_id") or doc.get("id"), doc) for doc in data
    )

    for cid, doc in items:
        for t in doc.get("extracts", []):
            if len(t) < 3:
                continue
            s = _clean_ws(str(t[0]))
            r = _clean_ws(str(t[1]))
            o = _clean_ws(str(t[2]))
            buf.append((cid, "b", _norm_exact(o), s, r, o))
            buf.append((cid, "c", _norm_exact(s), s, r, o))
            if len(buf) >= TRIPLE_DB_BATCH_SIZE:
                con.executemany("INSERT INTO triples VALUES (?,?,?,?,?,?)", buf)
                buf.clear()
    if buf:
        con.executemany("INSERT INTO triples VALUES (?,?,?,?,?,?)", buf)
    con.execute("INSERT INTO triple_db_ready VALUES (1)")
    con.commit()
    n = con.execute("SELECT COUNT(*) FROM triples").fetchone()[0]
    print(f"  Triple DB built: {n:,} rows  ({time.perf_counter()-t0:.1f}s)")


# ─────────────────────────────────────────────────────────────────────────────
# Skip set — SQLite-backed, in-memory for fast lookup
# ─────────────────────────────────────────────────────────────────────────────

def _init_skip_table(con: sqlite3.Connection) -> None:
    con.execute("""
        CREATE TABLE IF NOT EXISTS skip_combos (
            node_a TEXT NOT NULL,
            node_b TEXT NOT NULL,
            node_c TEXT NOT NULL,
            PRIMARY KEY (node_a, node_b, node_c)
        )
    """)
    con.commit()


def _load_skip(con: sqlite3.Connection) -> set:
    rows = con.execute(
        "SELECT node_a, node_b, node_c FROM skip_combos"
    ).fetchall()
    return set(rows)


def _add_skip(
    con: sqlite3.Connection,
    skip: set,
    node_a: str,
    node_b: str,
    node_c: str,
) -> None:
    key = (node_a, node_b, node_c)
    if key in skip:
        return
    skip.add(key)
    con.execute("INSERT OR IGNORE INTO skip_combos VALUES (?,?,?)", key)
    con.commit()


# ─────────────────────────────────────────────────────────────────────────────
# Convergent chain finder
# ─────────────────────────────────────────────────────────────────────────────

def _find_convergent_chain(
    triple_con: sqlite3.Connection,
    node_a: str,
    node_b: str,
    node_c: str,
    doc_a: dict,
) -> tuple[list, list, list] | None:
    """
    t1 = [s1, r1, s2]  from doc_A
    t2 = [s4, r2, s2]  from doc_B  (same s2)
    t3 = [s2, r3, s3]  from doc_C
    """
    for t1_raw in doc_a.get("extracts", []):
        if len(t1_raw) < 3:
            continue
        s1 = _clean_ws(str(t1_raw[0]))
        s2 = _clean_ws(str(t1_raw[2]))
        if not _entities_differ(s1, s2):
            continue
        s2_norm = _norm_exact(s2)

        rows = triple_con.execute("""
            SELECT b.s, b.r,
                   c.r, c.o
            FROM   triples b
            JOIN   triples c
                   ON  c.chunk_id = ?
                   AND c.role     = 'c'
                   AND c.key_norm = b.key_norm
            WHERE  b.chunk_id = ?
            AND    b.role     = 'b'
            AND    b.key_norm = ?
            LIMIT  50
        """, (node_c, node_b, s2_norm)).fetchall()

        for s4, r2, r3, s3 in rows:
            s4 = _clean_ws(s4)
            s3 = _clean_ws(s3)
            if not _entities_differ(s4, s2): continue
            if not _entities_differ(s4, s1): continue
            if not _entities_differ(s2, s3): continue
            if not _entities_differ(s1, s3): continue
            if not _entities_differ(s4, s3): continue
            return (
                [s1, _clean_ws(str(t1_raw[1])), s2],
                [s4, r2, s2],
                [s2, r3, s3],
            )

    return None


# ─────────────────────────────────────────────────────────────────────────────
# Component entry — precomputed matrices + sorted candidate list
# ─────────────────────────────────────────────────────────────────────────────

class _CompEntry:
    __slots__ = (
        "nodes", "node_set", "node_to_idx",
        "A_csr", "sub_deg",
        "pi", "pj",
        "scores",
        "order",
        "succ_sets",
    )

    def __init__(self, nodes, node_to_idx, A_csr, sub_deg,
                 pi, pj, scores, order, succ_sets):
        self.nodes       = nodes
        self.node_set    = frozenset(nodes)
        self.node_to_idx = node_to_idx
        self.A_csr       = A_csr
        self.sub_deg     = sub_deg
        self.pi          = pi
        self.pj          = pj
        self.scores      = scores
        self.order       = order
        self.succ_sets   = succ_sets


def _build_mink_convergent(
    A_csr,
    sub_deg: np.ndarray,
    pi: np.ndarray,
    pj: np.ndarray,
    succ_sets: list,
) -> np.ndarray:
    mink = np.full(len(pi), np.inf, dtype=np.float32)
    for r, (i, j) in enumerate(zip(pi, pj)):
        shared = succ_sets[i] & succ_sets[j]
        if shared:
            mink[r] = float(np.min(sub_deg[list(shared)]))
    return mink


def _build_component_entry(G, comp: set) -> "_CompEntry | None":
    """
    Build a _CompEntry for one weakly-connected component using the
    convergent A·Aᵀ matrix (shared successors) instead of the linear A²
    (2-hop reachability) matrix.
    """
    nodes = list(comp)
    n     = len(nodes)
    if n < 3:
        return None

    node_to_idx = {nd: i for i, nd in enumerate(nodes)}
    subgraph    = G.subgraph(comp)

    rows, cols = [], []
    for u, v in subgraph.edges():
        rows.append(node_to_idx[u])
        cols.append(node_to_idx[v])
    if not rows:
        return None

    A_csr = sp.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(n, n)
    )
    A_csr.setdiag(0)
    A_csr.eliminate_zeros()

    sub_undir = nx.Graph(subgraph)
    sub_deg   = np.array(
        [sub_undir.degree(nodes[i]) for i in range(n)], dtype=np.float32
    )

    succ_sets = [
        frozenset(A_csr.indices[A_csr.indptr[i]:A_csr.indptr[i + 1]].tolist())
        for i in range(n)
    ]

    # A·Aᵀ: entry (i,j) is nonzero iff nodes i and j share at least one
    # common successor — exactly the convergent topology condition.
    M = (A_csr @ A_csr.T).tocoo()
    M.setdiag(0)

    mask = np.asarray(M.row) < np.asarray(M.col)
    pi   = np.asarray(M.row)[mask]
    pj   = np.asarray(M.col)[mask]

    if len(pi) == 0:
        return None

    mink_vals = _build_mink_convergent(A_csr, sub_deg, pi, pj, succ_sets)

    valid_mask = mink_vals < np.inf
    if not np.any(valid_mask):
        return None

    pi        = pi[valid_mask]
    pj        = pj[valid_mask]
    mink_vals = mink_vals[valid_mask]

    scores = sub_deg[pi] + sub_deg[pj] + mink_vals
    order  = np.argsort(scores, kind="stable")

    return _CompEntry(
        nodes=nodes, node_to_idx=node_to_idx,
        A_csr=A_csr, sub_deg=sub_deg,
        pi=pi, pj=pj,
        scores=scores, order=order,
        succ_sets=succ_sets,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Component cache
# ─────────────────────────────────────────────────────────────────────────────

class ComponentCache:
    """
    Persistent cache of _CompEntry objects across iterations.

    Usage:
        cache = ComponentCache(G)        # builds entries for all components
        ...
        best = _pick_best_path(G, triple_con, cache, skip)
        ... PASS/FAIL updates G ...
        cache.invalidate_touched(G, [node_a, node_b, node_c])
    """

    def __init__(self, G):
        self._entries: dict[int, _CompEntry] = {}
        self._next_id = 0
        self._rebuild_all(G)

    def _rebuild_all(self, G) -> None:
        self._entries.clear()
        self._next_id = 0
        print("Building component cache (convergent) ...")
        t0    = time.perf_counter()
        comps = list(
            nx.weakly_connected_components(G)
            if G.is_directed() else nx.connected_components(G)
        )
        built = 0
        for comp in comps:
            entry = _build_component_entry(G, comp)
            if entry is not None:
                self._entries[self._next_id] = entry
                self._next_id += 1
                built += 1
        print(f"  {built:,} components cached  ({time.perf_counter()-t0:.1f}s)")

    def entries(self):
        return self._entries.values()

    def invalidate_touched(self, G, touched_nodes: list[str]) -> None:
        """
        Call after PASS/FAIL has mutated G. Removes any cached component
        entry whose node set intersects `touched_nodes`, then rebuilds
        fresh entries for whatever remains of those nodes' neighbourhoods.
        """
        touched_set = set(touched_nodes)
        stale_ids = [
            cid for cid, entry in self._entries.items()
            if entry.node_set & touched_set
        ]
        if not stale_ids:
            return
        surviving: set = set()
        for cid in stale_ids:
            for nd in self._entries[cid].nodes:
                if G.has_node(nd):
                    surviving.add(nd)
            del self._entries[cid]
        if not surviving:
            return
        sub = G.subgraph(surviving)
        new_comps = (
            nx.weakly_connected_components(sub)
            if sub.is_directed() else nx.connected_components(sub)
        )
        for comp in new_comps:
            entry = _build_component_entry(G, comp)
            if entry is not None:
                self._entries[self._next_id] = entry
                self._next_id += 1


# ─────────────────────────────────────────────────────────────────────────────
# TRUE GLOBAL MINIMUM path selection (convergent)
# ─────────────────────────────────────────────────────────────────────────────

def _pick_best_path(
    G,
    triple_con: sqlite3.Connection,
    comp_cache: ComponentCache,
    skip: set,
) -> dict | None:
    """
    Find the single highest-scoring valid convergent (A, B) → C path.
    Returns a path_data dict or None if no valid path exists.

    best_score always tracks the ACTUAL degree_sum of a verified, valid
    chain — not the optimistic lower-bound promised by mink_vals.
    """
    best_path  = None
    best_score = np.inf

    for entry in comp_cache.entries():
        nodes     = entry.nodes
        sub_deg   = entry.sub_deg
        pi, pj    = entry.pi, entry.pj
        scores    = entry.scores
        order     = entry.order
        succ_sets = entry.succ_sets

        for rank in order:
            promised_score = float(scores[rank])
            if promised_score >= best_score:
                break

            i_sub = int(pi[rank])
            j_sub = int(pj[rank])

            node_a = nodes[i_sub]
            node_b = nodes[j_sub]

            shared_c_idxs = succ_sets[i_sub] & succ_sets[j_sub]
            if not shared_c_idxs:
                continue

            c_arr    = np.array(list(shared_c_idxs), dtype=np.intp)
            c_sorted = c_arr[np.argsort(sub_deg[c_arr], kind="stable")]

            # Fetch doc_a from SQLite on demand
            doc_a = _fetch_doc(triple_con, node_a)

            for c_idx in c_sorted:
                c_idx  = int(c_idx)
                node_c = nodes[c_idx]

                actual_score = float(
                    sub_deg[i_sub] + sub_deg[j_sub] + sub_deg[c_idx]
                )
                if actual_score >= best_score:
                    break

                if (node_a, node_b, node_c) in skip:
                    continue

                chain = _find_convergent_chain(
                    triple_con, node_a, node_b, node_c, doc_a
                )

                if chain is None:
                    _add_skip(triple_con, skip, node_a, node_b, node_c)
                    continue

                best_score = actual_score
                best_path = {
                    "path"      : [node_a, node_b, node_c],
                    "degree_sum": int(actual_score),
                    "degree_a"  : int(sub_deg[i_sub]),
                    "degree_b"  : int(sub_deg[j_sub]),
                    "degree_c"  : int(sub_deg[c_idx]),
                    "chain"     : chain,
                    "node_a_id" : node_a,
                    "node_b_id" : node_b,
                    "node_c_id" : node_c,
                }
                break

    return best_path


# ─────────────────────────────────────────────────────────────────────────────
# Graph update helpers
# ─────────────────────────────────────────────────────────────────────────────

def _graph_remove_pass(G, node_a: str, node_b: str, node_c: str) -> int:
    """PASS: remove all 3 nodes and all their incident edges."""
    removed = 0
    for node in [node_a, node_b, node_c]:
        if not G.has_node(node):
            continue
        for src, dst in list(G.in_edges(node)) + list(G.out_edges(node)):
            if G.has_edge(src, dst):
                G.remove_edge(src, dst)
                removed += 1
        G.remove_node(node)
    return removed


def _graph_remove_fail(G, node_a: str, node_b: str, node_c: str) -> int:
    """FAIL: remove only edges A→C and B→C (convergent topology)."""
    removed = 0
    for src, dst in [(node_a, node_c), (node_b, node_c)]:
        if G.has_edge(src, dst):
            G.remove_edge(src, dst)
            removed += 1
    return removed


# ─────────────────────────────────────────────────────────────────────────────
# Question generation   (convergent)
# ─────────────────────────────────────────────────────────────────────────────

def _generate_q1(client, text: str, triple: list) -> tuple[str | None, str | None]:
    """Q1: mentions s1, answer = s2."""
    s1, r1, s2 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST mention "{s1}" by name.\n'
        f'  - The relation is "{r1}".\n'
        f'  - The correct answer is "{s2}" — do NOT include "{s2}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s2}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s2
    except Exception as e:
        print(f"    [Q1 error] {e}")
        return None, None


def _generate_q2(client, text: str, triple: list) -> tuple[str | None, str | None]:
    """Q2: mentions s4, answer = s2 (same bridge as Q1)."""
    s4, r2, s2 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST mention "{s4}" by name.\n'
        f'  - The relation is "{r2}".\n'
        f'  - The correct answer is "{s2}" — do NOT include "{s2}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s2}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s2
    except Exception as e:
        print(f"    [Q2 error] {e}")
        return None, None


def _generate_q3(client, text: str, triple: list) -> tuple[str | None, str | None]:
    """Q3: mentions s2 (bridge) verbatim, answer = s3."""
    s2, r3, s3 = triple
    prompt = (
        f"Text (use ONLY this — no external knowledge):\n{text}\n\n"
        f"Triple: {_fmt(triple)}\n\n"
        f"Task: Write a short, natural question.\n"
        f"Rules:\n"
        f'  - The question MUST mention "{s2}" by name (verbatim).\n'
        f'  - The relation is "{r3}".\n'
        f'  - The correct answer is "{s3}" — do NOT include "{s3}" in the question.\n'
        f"  - Answerable from the text above only.\n"
        f"  - Aim for under 15 words.\n\n"
        f'Return JSON only: {{"question": "...", "answer": "{s3}"}}'
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL, temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": "You are a question generation assistant. Return JSON only."},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), s3
    except Exception as e:
        print(f"    [Q3 error] {e}")
        return None, None


def _generate_final(
    client,
    q1: str, a1: str,
    q2: str, a2: str,
    q3: str, a3: str,
) -> tuple[str | None, str | None]:
    instruction = (
        "Generate a multi-hop question and answer based on the provided reasoning chain "
        "(Step 1, Step 2, Step 3) following a CONVERGING pattern where two independent "
        "reasoning paths both lead to the shared final answer.\n\n"
        "### Pattern: A→C and B→C (converging)\n"
        "- <1-hop>: Independent question about source A; answer a1\n"
        "- <2-hop>: Independent question about source B; answer a2\n"
        "- <3-hop>: Question about shared target C that requires BOTH a1 AND a2 to answer; final answer a3\n\n"
        "### Instructions:\n"
        "1. **Generate a Multi-Hop Question**: Form ONE natural question that explicitly requires "
        "combining the answers from BOTH <1-hop> AND <2-hop> to resolve <3-hop>. "
        "The question must be unanswerable without traversing both paths.\n"
        "2. **Preserve the Question Pattern from Step 3**: Extract the exact question type and structure from "
        f"<3-hop> query: '{q3}' (e.g., What is, Who, Where, When) and apply it to the multi-hop context. "
        "Do NOT use generic patterns like 'What entity', 'What organization', or 'What is the name of'.\n"
        "3. **Multi-Hop Reasoning Chain Tags**:\n"
        f"   - <1-hop>: Q: {q1} → A: {a1}\n"
        f"   - <2-hop>: Q: {q2} → A: {a2}\n"
        f"   - <3-hop>: Q: {q3} → A: {a3}\n"
        f"   - The final question must NOT directly name the intermediate answers '{a1}' or '{a2}'.\n"
        "4. **Descriptive Anchoring**: Use only descriptive context present in both "
        f"<1-hop> ('{q1}') and <2-hop> ('{q2}') questions as anchors. "
        "Do NOT introduce external knowledge.\n"
        f"5. **Final Answer Constraint**: The answer must be exactly: '{a3}'.\n\n"
        "### Avoid Generic Patterns:\n"
        "BAD: 'What entity is associated with...'\n"
        "BAD: 'What organization is connected to...'\n"
        "BAD: 'What is the name of...'\n"
        "BAD: 'What individual...'\n\n"
        "### Prefer Specific Patterns from <3-hop>:\n"
        "GOOD: 'What is the [specific concept from <3-hop>] that links [anchor from <1-hop>] and [anchor from <2-hop>]?'\n"
        "GOOD: 'Who [specific action from <3-hop>] connecting [anchor1] and [anchor2]?'\n"
        "GOOD: 'Where [specific location query from <3-hop>] involving both [anchor1] and [anchor2]?'\n\n"
        "### Output Format:\n"
        f'Return JSON only: {{"question": "multi-hop question preserving <3-hop> pattern", "answer": "{a3}"}}'
    )

    prompt = (
        f"CONVERGING reasoning chain (A→C and B→C):\n\n"
        f"<1-hop> Q: {q1} → A: {a1}\n"
        f"<2-hop> Q: {q2} → A: {a2}\n"
        f"<3-hop> Q: {q3} → A: {a3}\n\n"
        f"Following the instructions above, create a single natural multi-hop question that:\n"
        f"- Requires BOTH <1-hop> AND <2-hop> answers to reach <3-hop>\n"
        f"- Preserves the question pattern from <3-hop>: '{q3}'\n"
        f"- Does NOT directly mention intermediate answers '{a1}' or '{a2}'\n"
        f"- Uses only descriptive context from <1-hop> and <2-hop> as anchors\n"
        f"- Has exactly '{a3}' as the final answer"
    )

    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user",   "content": prompt},
            ],
        )
        result = _parse_json(resp.choices[0].message.content)
        return result.get("question"), a3
    except Exception as e:
        print(f"    [Final error] {e}")
        return None, None


# ─────────────────────────────────────────────────────────────────────────────
# Validators
# ─────────────────────────────────────────────────────────────────────────────

def _validate_q1(q: str, s1: str, s2: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s1).lower() not in ql:
        print(f"    [Q1 invalid] '{s1}' not found in question"); return False
    if _clean_ws(s2).lower() in ql:
        print(f"    [Q1 invalid] answer '{s2}' leaks"); return False
    return True


def _validate_q2(q: str, s4: str, s2: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s4).lower() not in ql:
        print(f"    [Q2 invalid] '{s4}' not found in question"); return False
    if _clean_ws(s2).lower() in ql:
        print(f"    [Q2 invalid] answer '{s2}' leaks"); return False
    return True


def _validate_q3(q: str, s2: str, s3: str) -> bool:
    if not q: return False
    ql = _clean_ws(q).lower()
    if _clean_ws(s2).lower() not in ql:
        print(f"    [Q3 invalid] bridge '{s2}' not found in question"); return False
    if _clean_ws(s3).lower() in ql:
        print(f"    [Q3 invalid] answer '{s3}' leaks"); return False
    return True


# ─────────────────────────────────────────────────────────────────────────────
# Generate: one path → question set
# ─────────────────────────────────────────────────────────────────────────────

def _generate_for_path(
    client,
    path_data: dict,
    triple_con: sqlite3.Connection,
) -> dict | None:
    """
    Generate Q1/Q2/Q3 + Final for a convergent (A, B → C) path.

    path_data keys (from _pick_best_path):
        path        : [node_a_id, node_b_id, node_c_id]
        chain       : (t1, t2, t3)  — each a 3-element list
        node_a_id   : str
        node_b_id   : str
        node_c_id   : str
        degree_sum  : int

    Chain semantics:
        t1 = [s1, r1, s2]   A → bridge        (Q1: mention s1, answer s2)
        t2 = [s4, r2, s2]   B → bridge        (Q2: mention s4, answer s2)
        t3 = [s2, r3, s3]   bridge → final    (Q3: mention s2, answer s3)
    """
    node_a_id = path_data["node_a_id"]
    node_b_id = path_data["node_b_id"]
    node_c_id = path_data["node_c_id"]
    chain     = path_data.get("chain")

    # ── Unpack chain ──────────────────────────────────────────────────────────
    if not chain or len(chain) < 3:
        print("  SKIP: malformed chain")
        return None

    t1_raw, t2_raw, t3_raw = chain
    t1 = [str(x).strip() for x in t1_raw[:3]] if t1_raw and len(t1_raw) >= 3 else []
    t2 = [str(x).strip() for x in t2_raw[:3]] if t2_raw and len(t2_raw) >= 3 else []
    t3 = [str(x).strip() for x in t3_raw[:3]] if t3_raw and len(t3_raw) >= 3 else []

    if not t1 or not t2 or not t3:
        print("  SKIP: one or more triples are empty")
        return None

    # t1=[s1,r1,s2]  t2=[s4,r2,s2]  t3=[s2,r3,s3]
    s1, r1, s2 = t1
    s4, r2, _  = t2   # bridge is s2 (same as t1[2])
    _,  r3, s3 = t3   # final answer

    print(f"  triple_1 (A→bridge) : {_fmt(t1)}")
    print(f"  triple_2 (B→bridge) : {_fmt(t2)}")
    print(f"  triple_3 (bridge→C) : {_fmt(t3)}")
    print(f"  bridge              : '{s2}'")
    print(f"  final answer        : '{s3}'")

    if not _entities_differ(s2, s3):
        print(f"  SKIP: bridge '{s2}' too similar to final answer '{s3}'")
        return None

    # ── Fetch doc texts from SQLite ───────────────────────────────────────────
    doc_a = _fetch_doc(triple_con, node_a_id)
    doc_b = _fetch_doc(triple_con, node_b_id)
    doc_c = _fetch_doc(triple_con, node_c_id)

    text_a = doc_a.get("text", "")
    text_b = doc_b.get("text", "")
    text_c = doc_c.get("text", "")

    # ── Generate sub-questions ────────────────────────────────────────────────
    # Q1: about node_A, must mention s1, answer = s2
    q1, a1 = _generate_q1(client, text_a, t1)
    if not _validate_q1(q1, s1, s2):
        print("  SKIP: Q1 failed")
        return None
    print(f"  Q1: {q1}\n  A1: {a1}")

    # Q2: about node_B, must mention s4, answer = s2
    q2, a2 = _generate_q2(client, text_b, t2)
    if not _validate_q2(q2, s4, s2):
        print("  SKIP: Q2 failed")
        return None
    print(f"  Q2: {q2}\n  A2: {a2}")

    # Q3: about node_C, must mention s2 (bridge), answer = s3
    q3, a3 = _generate_q3(client, text_c, t3)
    if not _validate_q3(q3, s2, s3):
        print("  SKIP: Q3 failed")
        return None
    print(f"  Q3: {q3}\n  A3: {a3}")

    # ── Generate final multi-hop question ─────────────────────────────────────
    final_q, final_a = _generate_final(client, q1, a1, q2, a2, q3, a3)
    print(f"\n  FINAL Q : {final_q}")
    print(f"  FINAL A : {a3}")

    return {
        "path"           : [node_a_id, node_b_id, node_c_id],
        "degree_sum"     : path_data.get("degree_sum"),
        "triple_1"       : t1,
        "triple_2"       : t2,
        "triple_3"       : t3,
        "entity_bridge"  : s2,
        "entity_final"   : s3,
        "reasoning_steps": [
            {"hop": 1, "triple": t1, "question": q1, "answer": a1,
             "source_chunk": node_a_id, "context": text_a},
            {"hop": 2, "triple": t2, "question": q2, "answer": a2,
             "source_chunk": node_b_id, "context": text_b},
            {"hop": 3, "triple": t3, "question": q3, "answer": a3,
             "source_chunk": node_c_id, "context": text_c},
        ],
        "final_question" : final_q,
        "final_answer"   : a3,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Judge
# ─────────────────────────────────────────────────────────────────────────────

_JUDGE_SYSTEM = """You are a judge for 3-hop multi-hop questions.

The sub-questions Q1/Q2/Q3 are provided as context only — do NOT judge them.
Your only job is to evaluate the FINAL QUESTION on TWO criteria:

1. MULTI-HOP REASONING — the Final Question must require 3 hops to answer correctly.
2. ANSWER CORRECTNESS — A3 must correctly answer the Final Question based on the provided reasoning steps.

Write one sentence of reasoning, then return JSON on the last line:
{"verdict": "pass"} or {"verdict": "fail", "reason": "..."}
"""


def _judge(client, record: dict) -> tuple[str, str]:
    steps = record.get("reasoning_steps") or []
    s0 = steps[0] if len(steps) > 0 else {}
    s1 = steps[1] if len(steps) > 1 else {}
    s2 = steps[2] if len(steps) > 2 else {}

    user_msg = (
        f"Q1 : {s0.get('question', '')}\n"
        f"A1 : {s0.get('answer', '')}\n\n"
        f"Q2 : {s1.get('question', '')}\n"
        f"A2 : {s1.get('answer', '')}\n\n"
        f"Q3 : {s2.get('question', '')}\n"
        f"A3 : {s2.get('answer', '')}\n\n"
        f"Final Q : {record.get('final_question', '')}\n"
        f"Final A : {record.get('final_answer', '')}\n\n"
        f"Judge on Multi-Hop Reasoning and Answer Correctness only.\n"
        f"One sentence of reasoning, then JSON on the final line."
    )

    def _try_judge():
        resp = client.chat.completions.create(
            model=JUDGE_MODEL, temperature=0.0,
            messages=[
                {"role": "system", "content": _JUDGE_SYSTEM},
                {"role": "user",   "content": user_msg},
            ],
        )
        content = resp.choices[0].message.content.strip()
        if not content:
            raise ValueError("Judge returned empty response")

        # Try every line from the bottom up, looking for valid JSON
        for ln in reversed(content.splitlines()):
            ln = ln.strip()
            if not ln:
                continue
            ln = re.sub(r'^```(?:json)?\s*|\s*```$', '', ln).strip()
            if not ln.startswith("{"):
                continue
            try:
                parsed  = json.loads(ln)
                verdict = parsed.get("verdict", "fail").lower()
                reason  = parsed.get("reason", "")
                if verdict not in ("pass", "fail"):
                    return "fail", "invalid verdict"
                return verdict, reason
            except json.JSONDecodeError:
                continue

        # Last resort: scan entire content for a JSON object
        match = re.search(r'\{[^{}]*"verdict"\s*:\s*"(pass|fail)"[^{}]*\}', content)
        if match:
            try:
                parsed  = json.loads(match.group())
                verdict = parsed.get("verdict", "fail").lower()
                reason  = parsed.get("reason", "")
                return verdict, reason
            except json.JSONDecodeError:
                pass

        raise ValueError(f"No valid JSON verdict found in judge response: {content!r}")

    for attempt in range(3):
        try:
            return _try_judge()
        except Exception as e:
            print(f"    [Judge attempt {attempt+1}/3 failed] {e}")
            time.sleep(1)

    return "fail", "judge error: all 3 attempts failed"


# ─────────────────────────────────────────────────────────────────────────────
# Output helpers
# ─────────────────────────────────────────────────────────────────────────────

def _write_readable(fh, record: dict, verdict: str, reason: str = "") -> None:
    path   = record.get("path", [])
    steps  = record.get("reasoning_steps", [])
    fh.write(f"\n[{verdict.upper()}]  {' / '.join(str(p) for p in path)}\n")
    fh.write(f"entity_bridge      : {record.get('entity_bridge','')}\n")
    fh.write(f"entity_final       : {record.get('entity_final','')}\n")
    fh.write(f"degree_sum         : {record.get('degree_sum','')}\n")
    if reason:
        fh.write(f"reason             : {reason}\n")
    fh.write("\nReasoning steps:\n")
    for step in steps:
        fh.write(f"  Hop {step.get('hop','')} [{step.get('source_chunk','')}]\n")
        fh.write(f"  Q: {step.get('question','')}\n")
        fh.write(f"  A: {step.get('answer','')}\n\n")
    fh.write(f"Final Q : {record.get('final_question','')}\n")
    fh.write(f"Final A : {record.get('final_answer','')}\n")
    fh.write(("=" if verdict == "pass" else "─") * 60 + "\n")
    fh.flush()


def _load_done_keys(all_results_jsonl: str) -> set[tuple]:
    done = set()
    if not os.path.exists(all_results_jsonl):
        return done
    with open(all_results_jsonl, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                key = tuple(rec.get("path", []))
                if key:
                    done.add(key)
            except Exception:
                pass
    return done


# ─────────────────────────────────────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(output_folder: str = OUTPUT_FOLDER) -> list[dict]:
    """
    Full unified pipeline: greedy-select → generate → judge → update graph.
    Fully resumable.

    Uses:
      - Startup cache (G only) to skip slow graph loading on restarts.
      - SQLite triple_index.db for doc lookup and triple matching.
      - SQLite-backed skip set: persists across restarts, no RAM blowup.
      - ComponentCache (convergent A·Aᵀ) to skip redundant recomputation.

    Returns list of accepted (PASS) question records loaded from disk
    (not kept in RAM during the run).
    """
    t0     = time.perf_counter()
    client = OpenAI()

    results_folder   = os.path.join(output_folder, "results")
    questions_folder = os.path.join(output_folder, "questions")
    os.makedirs(results_folder,   exist_ok=True)
    os.makedirs(questions_folder, exist_ok=True)

    triple_db_path       = os.path.join(results_folder,   "triple_index.db")
    filtered_jsonl       = os.path.join(questions_folder, "multihop_questions_filtered.jsonl")
    filtered_readable    = os.path.join(questions_folder, "multihop_questions_filtered_readable.txt")
    all_results_jsonl    = os.path.join(questions_folder, "multihop_questions_all_results.jsonl")
    all_results_readable = os.path.join(questions_folder, "multihop_questions_all_results_readable.txt")

    # ── Open DB first ─────────────────────────────────────────────────────────
    print("\n=== Startup ===")
    triple_con = _open_db(triple_db_path)

    # ── Load G (from cache or pkl) ────────────────────────────────────────────
    G = _load_startup_cache(output_folder)
    if G is None:
        G, data = _load_graph_and_data(output_folder)
        print("\n=== Phase 0a: Doc DB ===")
        _build_doc_db(data, triple_con)
        print("\n=== Phase 0b: Triple DB ===")
        _build_triple_db(data, triple_con)
        del data   # free raw data — it's now in SQLite
        _save_startup_cache(output_folder, G)
    else:
        print("\n=== Phase 0a: Doc DB ===")
        _build_doc_db(None, triple_con)
        print("\n=== Phase 0b: Triple DB ===")
        _build_triple_db(None, triple_con)

    # ── Skip table ────────────────────────────────────────────────────────────
    _init_skip_table(triple_con)
    skip: set = _load_skip(triple_con)
    print(f"  Skip set loaded: {len(skip):,} persisted combos")

    # ── Resume: reconstruct graph state from prior runs ───────────────────────
    done_keys      = _load_done_keys(all_results_jsonl)
    accepted_count = 0

    if done_keys:
        print(f"\n[Resume] {len(done_keys)} paths already judged — replaying graph updates ...")
        removed_total = 0
        with open(all_results_jsonl, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec     = json.loads(line)
                    verdict = rec.get("verdict", "fail")
                    path    = rec.get("path", [])
                    if len(path) == 3:
                        na, nb, nc = path
                        if verdict == "pass":
                            removed_total += _graph_remove_pass(G, na, nb, nc)
                            accepted_count += 1
                        else:
                            removed_total += _graph_remove_fail(G, na, nb, nc)
                except Exception:
                    pass
        print(f"  Graph edges removed during replay: {removed_total}")
        print(f"  Accepted so far: {accepted_count}")

    # ── Build ComponentCache AFTER resume replay ──────────────────────────────
    print("\n=== Building component cache ===")
    comp_cache = ComponentCache(G)

    # ── Phase 2: generate → judge loop ───────────────────────────────────────
    print("\n=== Phase 2: Generate → Judge loop ===")
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for fpath, header in [
        (filtered_readable,
         f"FILTERED MULTI-HOP QUESTIONS (PASS ONLY)\nStarted: {now}\n{'='*60}\n"),
        (all_results_readable,
         f"ALL MULTI-HOP QUESTIONS — JUDGE RESULTS\nStarted: {now}\n{'='*60}\n"),
    ]:
        if not os.path.exists(fpath):
            with open(fpath, "w", encoding="utf-8") as fh:
                fh.write(header)

    failed_count = 0
    gen_failed   = 0
    iteration    = 0

    with open(filtered_jsonl,       "a", encoding="utf-8") as f_pass, \
         open(all_results_jsonl,    "a", encoding="utf-8") as f_all,  \
         open(filtered_readable,    "a", encoding="utf-8") as f_pass_txt, \
         open(all_results_readable, "a", encoding="utf-8") as f_all_txt:

        while True:
            iteration += 1
            if MAX_PATHS and (accepted_count + failed_count) >= MAX_PATHS:
                print(f"\nReached MAX_PATHS={MAX_PATHS} — stopping.")
                break

            print(f"\n{'='*70}")
            print(f"ITERATION {iteration}  |  "
                  f"Accepted: {accepted_count}  "
                  f"Failed: {failed_count}  "
                  f"Gen-failed: {gen_failed}")
            print(f"Graph: {G.number_of_nodes():,} nodes  "
                  f"{G.number_of_edges():,} edges  "
                  f"Skip: {len(skip):,}  |  "
                  f"Cached components: {len(list(comp_cache.entries())):,}")
            print(f"{'='*70}")

            # 1. Pick best convergent path
            path_data = _pick_best_path(G, triple_con, comp_cache, skip)
            if path_data is None:
                print("\nNo more valid paths in graph — pipeline complete.")
                break

            node_a, node_b, node_c = path_data["path"]
            print(f"\nSelected: {node_a}, {node_b} → {node_c}  "
                  f"(degree_sum={path_data['degree_sum']}  "
                  f"a={path_data['degree_a']}  "
                  f"b={path_data['degree_b']}  "
                  f"c={path_data['degree_c']})")

            # 2. Generate questions
            try:
                record = _generate_for_path(client, path_data, triple_con)
            except Exception as e:
                print(f"  Generation error: {e}")
                traceback.print_exc()
                record = None

            if record is None:
                gen_failed += 1
                _graph_remove_fail(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print("  ✗ Generation failed — removed edges A→C and B→C")
                continue

            # 3. Judge
            try:
                verdict, reason = _judge(client, record)
            except Exception as e:
                traceback.print_exc()
                verdict, reason = "fail", f"judge exception: {e}"

            print(f"\n  VERDICT: {verdict.upper()}"
                  + (f"  — {reason}" if reason else ""))

            # 4. Persist result
            rec_out                 = dict(record)
            rec_out["verdict"]      = verdict
            rec_out["judge_reason"] = reason

            f_all.write(json.dumps(rec_out, ensure_ascii=False) + "\n")
            f_all.flush()
            _write_readable(f_all_txt, record, verdict, reason)

            if verdict == "pass":
                accepted_count += 1
                f_pass.write(json.dumps(record, ensure_ascii=False) + "\n")
                f_pass.flush()
                _write_readable(f_pass_txt, record, "pass")
                _graph_remove_pass(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print(f"  ✓ PASS — removed nodes {node_a}, {node_b}, {node_c} + incident edges")
            else:
                failed_count += 1
                _graph_remove_fail(G, node_a, node_b, node_c)
                comp_cache.invalidate_touched(G, [node_a, node_b, node_c])
                print(f"  ✗ FAIL — removed edges A→C and B→C only")

    triple_con.close()

    elapsed = time.perf_counter() - t0
    print(f"\n{'='*70}")
    print(f"PIPELINE COMPLETE")
    print(f"  Iterations     : {iteration - 1}")
    print(f"  Accepted (PASS): {accepted_count}")
    print(f"  Rejected (FAIL): {failed_count}")
    print(f"  Gen-failed     : {gen_failed}")
    print(f"  Time           : {elapsed:.1f}s")
    print(f"\nOutputs:")
    print(f"  Filtered JSONL    : {filtered_jsonl}")
    print(f"  Filtered readable : {filtered_readable}")
    print(f"  All results JSONL : {all_results_jsonl}")
    print(f"  All results txt   : {all_results_readable}")
    print(f"{'='*70}")

    return []   # accepted records are in filtered_jsonl — not kept in RAM


## 5. Execute the configured stages


In [ ]:
# =============================================================================
# RUN
# =============================================================================

cleaned_path = OUTPUT_DIR / CLEANED_FILENAME

if RUN_CLEANING:
    clean_dataset(
        input_source=INPUT_DATA,
        output_path=cleaned_path,
        field_map=FIELD_MAP,
        recursive=RECURSIVE_INPUT,
    )

if RUN_GRAPH_BUILDING:
    G, data, edge_details, graph_file = create_graph_and_nodes(
        cleaned_file=cleaned_path,
        output_folder=OUTPUT_DIR,
    )

if RUN_LLM_PIPELINE:
    accepted_records = run_pipeline(output_folder=str(OUTPUT_DIR))
else:
    print(
        "\nLLM generation skipped (RUN_LLM_PIPELINE=False). "
        "Set it to True in the Configuration cell when you are ready."
    )
